# Корутины

Корутина (или сопрограмма) — это функция, которую можно приостановить, а затем возобновить её выполнение. В отличие от обычных функций, которые выполняются от начала до конца без остановок, корутина может неоднократно приостанавливаться и продолжать работу с места остановки, пока не дойдет до своего завершения. 

Как раз этому механизму работы корутин асинхронные программы и обязаны своей высокой производительностью даже в рамках одного потока операционной системы. Пока одна корутина приостановилась и ожидает, в это время включается другая корутина, которая также будет выполняться до приостановки. То есть программа в целом не простаивает, как это было бы с синхронным вариантом.

Принцип работы корутин схож с генераторами: корутина так же может отдавать какой-то объект, может принимать какой-то объект, при этом после каждого шага приостанавливает выполнение до тех пор, пока ей не прикажут продолжить. Только в более современных версиях Python (3.5+) нам не нужно писать порой громоздкие и запутанные yield-выражение. Вместо это asyncio работает с “нативными корутинами”, которые объявляются с использованием ключевых слов async и await.

In [17]:
import asyncio

async def coroutine():
    print("Шаг 1")
    await asyncio.sleep(1)  # Приостановка
    print("Шаг 2")

* asyncio.sleep(seconds) - позволяет программе уснуть на указанное количество секунд. А точнее, в терминах асинхронности, приостанавливает выполнение текущей корутины на указанное количество секунд. Функция очень полезна, и часто используется разработчиками. Например в тестах, чтобы сэмулировать какую нибудь IO-операцию, в ожидании которой корутина приостанавливает свое выполнение. Мы также будем её использовать в демонстрационных примерах.

* конструкций await внутри корутины может и не быть, но тогда теряется смысл этой самой корутины, ведь раз нам не нужно её приостанавливать, то с таким же успехом можно было объявить обычную функцию

## Awaitable объекты

Какие объекты в python можно ожидать(await) и почему?
Ответ: оператор await можно использовать только с так называемыми “awaitable”- объектами

Awaitable (ожидаемый объект) - технически, это любой объект, в котором реализован метод __ await __()

Корутины, что мы создаем с помощью конструкции async def, как раз являются awaitable-объектами.

In [14]:
async def main():
    pass

main_coro = main()
print(hasattr(main_coro, "__await__"))  # True
print(main_coro.__await__)  # <method-wrapper '__await__' of coroutine object at 0x100971010>

True
<method-wrapper '__await__' of coroutine object at 0x00000162FE6A7CC0>


## Запуск корутин

Корутина возвращает coroutine object 

In [18]:
async def coro():
    print("Executing")
    return 1

result = coro()
print(result) # <coroutine object coro at 0x1024f9d80>

<coroutine object coro at 0x00000162FE44ED40>


Корутины можно запускать через функцию asyncio.run(coroutine_object), передав в аргументы объект корутины.

asyncio.run вернет результат выполненной корутины (либо вернет None, если запускаемая корутина явно значений не возвращает). Функция asyncio.run(...) обычно является точкой входа в программу: она запускает основную корутину, которая в свою очередь уже порождает другие корутины, если это нужно.

In [31]:
import asyncio

async def coro():
    print("Executing")
    return 1

coro_obj = coro() # Создали объект корутины
try:
    result = asyncio.run(coro_obj) # Передали его в функцию запуска
    print(result)
except RuntimeError as e:
    print(e)

asyncio.run() cannot be called from a running event loop


Ошибка 'asyncio.run() cannot be called from a running event loop' возникает потому, что в Jupyter Notebook уже работает собственный event loop (цикл событий) для асинхронных операций. Вызов asyncio.run() пытается создать новый цикл событий, что запрещено, если один уже активен

In [ ]:
async def coro():
    print("Executing")
    return 1

result = await coro()
print(result)

Executing
None


Из корутины можно вызывать другие вложенные корутины (также, как и из обычной python-функции можно вызывать другие вложенные функции). Делается это через конструкцию await. Конструкция await позволяет запустить вложенную корутину и вернуть результат ее выполнения (вернет None, если вложенная корутина значений явно не возвращает).

При этом await как бы сигнализирует нам о том, что внутри этой вложенной корутины должна быть приостановка выполнения. Поэтому правильней сказать, await позволяется “дождаться” результата работы вложенной корутины.

## Пример

Обычный последовательный код, но написанный с помощью корутин

In [ ]:
import asyncio

async def sub_coro():
  print("Sub coroutine executing")
  await asyncio.sleep(0.1)
  print("Sub coroutine stopping")
  return 1

async def coro():
  print("Executing coro")


  # await сигнализирует о том, что внутри sub_coro 
  # будет приостановка выполнения
  result_of_sub_coro = await sub_coro()
  print("coro stopping")
  return result_of_sub_coro

result = await coro()
print(result)

Executing coro
Sub coroutine executing
Sub coroutine stopping
coro stopping
1


In [ ]:
import time

def sub_func():
  print("Sub func executing")
  time.sleep(0.1)
  print("Sub func stopping")
  return 1

def func():
  print("Executing func")
  result_of_sub_func = sub_func()
  print("func stopping")
  return result_of_sub_func

result = func()
print(result)

Executing func
Sub func executing
Sub func stopping
func stopping
1


Так где же обещанное конкурентное выполнение корутин и прирост в производительности? Это, по сути, тот же синхронный код, только со словами async и await.

Ответ: Конкурентное выполнение корутин в asyncio обеспечивает цикл событий (event loop). Он как раз и позволит нам добиться в разы лучшей производительности, в сравнении с синхронными программами.

Прирост производительности достигается за счёт того, что event loop эффективно управляет запуском корутин, минимизируя время простоя программы в целом. Происходит это примерно так: когда одна корутина простаивает, ожидая (await) выполнения какой-либо IO-операции, event loop запускает другую, уже готовую работать. Таким образом происходит “уплотнение” рабочего времени, и вся программа, состоящая из нескольких таких конкурентных корутин, в целом завершается быстрее, чем завершился бы её синхронный аналог.